# Chapter 12 &mdash; The Formal PDA $(Q,\Sigma,\Gamma,\Delta,q_0,z_0,F)$

**Concept 4 of the Chapter 12 decomposition:** *The Formal PDA $(Q,\Sigma,\Gamma,\Delta,q_0,z_0,F)$*

Seven components; $\Delta$ maps state &times; (input or $\varepsilon$) &times; stack symbol to a <i>set</i> of (state, push-string).

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter12/Concept-Formal-PDA/Concept-Formal-PDA.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


$$P = (Q,\Sigma,\Gamma,\Delta,q_0,z_0,F)$$

* $Q$ &mdash; finite states;
* $\Sigma$ &mdash; input alphabet;
* $\Gamma$ &mdash; **stack** alphabet, a separate alphabet from $\Sigma$;
* $\Delta: Q\times\Sigma_\varepsilon\times\Gamma \to \mathcal{P}(Q\times\Gamma^*)$;
* $q_0$ &mdash; start state;
* $z_0 \in \Gamma$ &mdash; the **initial stack symbol**;
* $F \subseteq Q$ &mdash; final states.

Three details worth noticing. $\Delta$ returns a **set**, so the machine is
nondeterministic. It returns a **string** to push, so one move can push any bounded
number of symbols. And $\Gamma$ is **separate from** $\Sigma$ &mdash; the stack may use
symbols that never appear in the input, which is often the cleanest design.

## 2. Definitions

### A machine using its own stack alphabet

# --- a thin wrapper over Jove's PDA runner -----------------------------
# run_pda returns (surviving-IDs, accepting-paths, visited-IDs); a string
# is accepted exactly when the list of accepting paths is non-empty.
def pda_accepts(P, s, acceptance='ACCEPT_F', STKMAX=6):
    surv, paths, visited = run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)
    return len(paths) > 0

def pda_npaths(P, s, acceptance='ACCEPT_F', STKMAX=6):
    return len(run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)[1])

In [ ]:
AnBn = md2mc('''PDA
!! Gamma = {#, A} -- 'A' never appears in the input.
I : a , # ; A#  -> I
I : a , A ; AA  -> I
I : b , A ; ''  -> M
M : b , A ; ''  -> M
M : '' , # ; #  -> F
I : '' , # ; #  -> F     !! the empty string
''')

# --- a thin wrapper over Jove's PDA runner -----------------------------
# run_pda returns (surviving-IDs, accepting-paths, visited-IDs); a string
# is accepted exactly when the list of accepting paths is non-empty.
def pda_accepts(P, s, acceptance='ACCEPT_F', STKMAX=6):
    surv, paths, visited = run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)
    return len(paths) > 0

def pda_npaths(P, s, acceptance='ACCEPT_F', STKMAX=6):
    return len(run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)[1])

### Reading the seven components off the dictionary

In [ ]:
def show_pda(P):
    for k in ['Q', 'Sigma', 'Gamma', 'q0', 'z0', 'F']:
        v = P[k]
        print("%-6s : %s" % (k, sorted(v) if isinstance(v, set) else v))
    print("Delta :")
    for (q, i, s), outs in sorted(P["Delta"].items()):
        print("   (%s, %r, %s) -> %s" % (q, i, s, sorted(outs)))

## 3. Tests

All seven components.

In [ ]:
show_pda(AnBn)
assert AnBn["z0"] in AnBn["Gamma"]

$\Gamma$ is **separate** from $\Sigma$: the stack symbol `A` is not an input symbol.

In [ ]:
print("Sigma :", sorted(AnBn["Sigma"]))
print("Gamma :", sorted(AnBn["Gamma"]))
print("Gamma - Sigma :", sorted(AnBn["Gamma"] - AnBn["Sigma"]))
assert 'A' in AnBn["Gamma"] and 'A' not in AnBn["Sigma"]

$\Delta$ returns a **set** of (state, push-string) pairs.

In [ ]:
for k, outs in sorted(AnBn["Delta"].items()):
    assert isinstance(outs, set)
    for (q2, push) in outs:
        assert isinstance(push, str)
print("every Delta value is a set of (state, push-string) pairs")

And the machine works.

In [ ]:
def in_anbn(s):
    k = len(s) - len(s.lstrip('a'))
    return s == 'a'*k + 'b'*(len(s)-k) and k == len(s)-k
from itertools import product
strs = [''.join(p) for k in range(7) for p in product('ab', repeat=k)]
bad = [s for s in strs if pda_accepts(AnBn, s, STKMAX=9) != in_anbn(s)]
print("mismatches over %d strings :" % len(strs), bad)
assert not bad

The push-string lets one move push several symbols.

In [ ]:
Two = md2mc('''PDA
I : a , # ; XY#  -> I
I : b , X ; ''   -> I
I : b , Y ; ''   -> I
I : '' , # ; #   -> F
''')
print("accepts 'abb' ?", pda_accepts(Two, 'abb', STKMAX=6))
assert pda_accepts(Two, 'abb', STKMAX=6)
print("one 'a' pushed TWO symbols, so two 'b's are needed to clear them.")

## 4. Exercises


1. Why is $z_0$ part of the definition rather than just "start with an empty stack"?
2. What is the largest push-string in your own PDA designs? Does it matter?
3. Write the DFA five-tuple as a PDA seven-tuple. What do you set $\Gamma$ to?

In [ ]:
# Your work for the exercises above.